# α-quartz SiO₂: training, dielectric response, Hessian, and spectra

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/02_sio2_workflow.ipynb)

This standalone notebook combines a compact response-model training demonstration, single-point electronic and ionic dielectric analysis, and zero-field molecular dynamics at **300 K**. The 72-atom QE-labeled structures came from classical MD trajectories at 300 K and 600 K, not AIMD. The tutorial subset preserves zero-field energy, force, stress, polarization, BEC, and electronic-susceptibility labels.

The training section uses only a 24/8 subset. Later analysis uses the bundled specialist checkpoint and the full response dataset; it does not depend on a checkpoint produced by the compact training demonstration. To keep the first MD run short, it defaults to 1,000 steps (0.5 ps at 0.5 fs), giving coarse raw spectral spacing near 67 cm⁻¹. Increase `MD_STEPS` in the MD cell or set `MACEFIELD_SIO2_MD_STEPS` for longer trajectories; 20,000 steps is 10 ps and takes about 20 times longer for the MD portion. The cell prints a rough estimate after its first 20 steps. A fixed-cell relaxation runs before MD, and the frequency-dependent curves remain exploratory.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import warnings
import logging

warnings.filterwarnings("ignore")

class _MACEConversionWarningFilter(logging.Filter):
    def filter(self, record):
        return not record.getMessage().startswith("Default dtype ")

logging.getLogger().addFilter(_MACEConversionWarningFilter())
os.environ["PYTHONWARNINGS"] = "ignore"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def ensure_checkout(path, url):
    path = Path(path)
    if (path / ".git").is_dir():
        branch = subprocess.run(
            ["git", "-C", str(path), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch == "main":
            return
        subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
        subprocess.run(
            ["git", "-C", str(path), "checkout", "-B", "main", "origin/main"],
            check=True,
        )
        return
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", url, str(path)],
        check=True,
    )


if IN_COLAB:
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    PACKAGE_REPO = Path("/content/mace-field")
    ensure_checkout(TUTORIAL_REPO, TUTORIAL_URL)
    ensure_checkout(PACKAGE_REPO, PACKAGE_URL)
else:
    CWD = Path.cwd().resolve()
    TUTORIAL_REPO = next(
        (
            path
            for path in (CWD, *CWD.parents)
            if (path / "tutorials/macefield/data").is_dir()
            and (path / "README.md").is_file()
        ),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run this notebook from a clone of mdi-group/mace-field-tutorial, "
            "or open it in Colab."
        )

    requested_repo = os.environ.get("MACEFIELD_REPO")
    if requested_repo:
        PACKAGE_REPO = Path(requested_repo).expanduser().resolve()
        if not (PACKAGE_REPO / "mace/modules/extensions.py").is_file():
            raise RuntimeError("MACEFIELD_REPO does not point to a MACE-Field checkout.")
        branch = subprocess.run(
            ["git", "-C", str(PACKAGE_REPO), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch != "main":
            raise RuntimeError("Set MACEFIELD_REPO to a checkout on the main branch.")
    else:
        candidates = [
            TUTORIAL_REPO.parent / "mace-field-main",
            TUTORIAL_REPO.parent / "mace-field",
        ]
        PACKAGE_REPO = None
        for candidate in candidates:
            if not (candidate / "mace/modules/extensions.py").is_file():
                continue
            branch = subprocess.run(
                ["git", "-C", str(candidate), "branch", "--show-current"],
                capture_output=True,
                text=True,
                check=False,
            ).stdout.strip()
            if branch == "main":
                PACKAGE_REPO = candidate.resolve()
                break
        if PACKAGE_REPO is None:
            PACKAGE_REPO = Path("/tmp/mace-field-main")
            ensure_checkout(PACKAGE_REPO, PACKAGE_URL)

os.environ["MACEFIELD_REPO"] = str(PACKAGE_REPO)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, str(PACKAGE_REPO))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--editable", str(PACKAGE_REPO)],
    check=True,
)
os.chdir(TUTORIAL_REPO)

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA_DIR = TUTORIAL_DIR / "data"


import hashlib
import urllib.request

MH1_MODEL_URL = "https://github.com/mdi-group/mace-field/releases/download/1.1.0/MACEField-MH-1.model"
MH1_MODEL_SHA256 = "3e654f1cd043df8df68002f7d7eec626b3e9dce7763dedbbf3710188bff048d0"

def _sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def select_md_model(source, provided_model):
    """Choose the bundled model, this notebook's trained model, or MH-1 pt_head."""
    source = source.strip().lower()
    head = None
    if source == "provided":
        model_path = Path(provided_model)
    elif source == "trained":
        model_path = globals().get("TRAINED_MODEL_PATH")
        if model_path is None or not Path(model_path).is_file():
            raise FileNotFoundError(
                "Run this notebook's training cell first, or choose MD_MODEL_SOURCE = 'provided' or 'mh1'."
            )
        model_path = Path(model_path)
    elif source == "mh1":
        model_path = TUTORIAL_DIR / "runs" / "foundation_assets" / "MACEField-MH-1.model"
        model_path.parent.mkdir(parents=True, exist_ok=True)
        if not model_path.is_file() or _sha256_file(model_path) != MH1_MODEL_SHA256:
            download_path = model_path.with_suffix(".download")
            print("Downloading MACEField-MH-1 for MD...", flush=True)
            urllib.request.urlretrieve(MH1_MODEL_URL, download_path)
            if _sha256_file(download_path) != MH1_MODEL_SHA256:
                download_path.unlink(missing_ok=True)
                raise ValueError("MACEField-MH-1 checksum verification failed.")
            download_path.replace(model_path)
        head = "pt_head"
    else:
        raise ValueError("MD_MODEL_SOURCE must be 'provided', 'trained', or 'mh1'.")

    if not model_path.is_file():
        raise FileNotFoundError(f"Model file does not exist: {model_path}")
    return model_path, head


print("Tutorial repository:", TUTORIAL_REPO)
print("MACE-Field source:", PACKAGE_REPO)


def run_streamed(command, *, cwd):
    """Stream a command's stdout and stderr into the notebook cell."""
    child_env = os.environ.copy()
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["PYTHONWARNINGS"] = "ignore"
    print("Running command:", command, flush=True)
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=child_env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


## 1. Inspect and fit the quartz response subset

Both source temperatures are represented in training and validation. Validation frames come from later blocks of each trajectory to reduce adjacent-frame leakage. The short fit demonstrates MACEField data wiring and response losses; it is not a converged quartz model.


In [ ]:
import numpy as np
from ase.io import read

TRAIN = DATA_DIR / "sio2/quartz_qe_train_sample.extxyz"
VALID = DATA_DIR / "sio2/quartz_qe_valid_sample.extxyz"
train = read(TRAIN, index=":")
valid = read(VALID, index=":")
frames = train + valid
fields = np.asarray([a.info["REF_electric_field"] for a in frames], dtype=float)
print("train / validation frames:", len(train), "/", len(valid))
print("atoms per frame:", sorted({len(a) for a in frames}))
print("field vectors:", np.unique(np.round(fields, 10), axis=0))
print("info labels:", sorted(k for k in frames[0].info if k.startswith("REF_")))
print("array labels:", sorted(k for k in frames[0].arrays if k.startswith("REF_")))


## Run a compact training demonstration

The default is five epochs on CUDA and one on CPU. Set `MACEFIELD_TUTORIAL_EPOCHS` before training for a longer fit. The MD cell can use the resulting model with `MD_MODEL_SOURCE = "trained"`; its other choices are the bundled model (default) and MH-1 (`pt_head`). The bundled 24/8 subset keeps the run small while retaining both source temperatures; independent validation is needed for model-quality claims.


In [ ]:
import os
import time
import yaml
import torch
import subprocess
import sys

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEFAULT_EPOCHS = 5 if DEVICE == "cuda" else 1
EPOCHS = int(os.environ.get("MACEFIELD_TUTORIAL_EPOCHS", DEFAULT_EPOCHS))
RUN_DIR = TUTORIAL_DIR / "runs" / f"sio2_qe_sample_{time.strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

config = {
    "name": "MACEField-alpha-quartz-QE-sample",
    "model": "MACEField",
    "loss": "universal_field",
    "train_file": str(TRAIN),
    "valid_file": str(VALID),
    "valid_fraction": 0,
    "E0s": "average",
    "seed": 23,
    "energy_key": "REF_energy",
    "forces_key": "REF_forces",
    "stress_key": "REF_stress",
    "electric_field_key": "REF_electric_field",
    "polarization_key": "REF_polarization",
    "becs_key": "REF_becs",
    "polarizability_key": "REF_polarizability",
    "compute_forces": True,
    "compute_stress": True,
    "compute_polarization": True,
    "compute_becs": True,
    "compute_polarizability": True,
    "energy_weight": 1.0,
    "forces_weight": 50.0,
    "stress_weight": 1.0,
    "polarization_weight": 10.0,
    "becs_weight": 100.0,
    "polarizability_weight": 200.0,
    "num_channels": 32,
    "num_interactions": 2,
    "max_L": 1,
    "r_max": 5.0,
    "batch_size": 1,
    "valid_batch_size": 1,
    "num_workers": 0,
    "lr": 0.005,
    "max_num_epochs": EPOCHS,
    "patience": max(3, EPOCHS),
    "eval_interval": 1,
    "error_table": "PerAtomFieldRMSE",
    "default_dtype": "float32",
    "device": DEVICE,
    "model_dir": str(RUN_DIR / "models"),
    "checkpoints_dir": str(RUN_DIR / "checkpoints"),
    "log_dir": str(RUN_DIR / "logs"),
    "results_dir": str(RUN_DIR / "results"),
    "work_dir": str(RUN_DIR / "work"),
}
for output_key in ("model_dir", "checkpoints_dir", "log_dir", "results_dir", "work_dir"):
    Path(config[output_key]).mkdir(parents=True, exist_ok=True)
CONFIG = RUN_DIR / "train.yml"
CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print("device:", DEVICE, "| epochs:", EPOCHS)
print("configuration:", CONFIG)

print(
    "Streaming per-epoch validation loss and energy, force, stress, polarization, "
    "BEC, and polarizability RMSEs (Hessian RMSE is None for this dataset).",
    flush=True,
)
run_streamed(
    [sys.executable, "-m", "mace.cli.run_train", "--config", str(CONFIG)],
    cwd=PACKAGE_REPO,
)
TRAINED_MODEL_PATH = Path(config["model_dir"]) / f"{config['name']}.model"
print("Model available for MD_MODEL_SOURCE = 'trained':", TRAINED_MODEL_PATH)


## What this fit does and does not show

The fit uses zero-field QE energy, force, stress, polarization, BEC, and polarizability labels sampled from both temperature trajectories. The structure configurations were generated by classical MD before their QE response calculations. This short subsample demonstrates MACEField training; it is not a converged SiO₂ potential. The MD cell defaults to the bundled checkpoint and can optionally use this notebook’s trained model or MACEField-MH-1 (`pt_head`). Increase `MACEFIELD_TUTORIAL_EPOCHS` before training for a longer local fit.

## 2. Single-point dielectric response, BECs, and Hessian

This section relaxes one structure at fixed cell with the bundled full-data checkpoint, evaluates polarization, BECs and electronic susceptibility, then differentiates the energy twice with respect to Cartesian positions.


In [ ]:
import numpy as np
import torch
from ase.io import read
from ase.optimize import BFGS
from mace.calculators import MACECalculator
from mace.tools import torch_tools

MODEL = DATA_DIR / "sio2/MACE-field-SiO2.model"
SIO2_DATA = DATA_DIR / "sio2/SiO2-preprocessed.extxyz"
atoms = read(SIO2_DATA, index=0)
atoms.info.clear()
for key in list(atoms.arrays):
    if key not in {"numbers", "positions"}:
        atoms.arrays.pop(key)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
relax_calc = MACECalculator(
    model_paths=str(MODEL),
    model_type="MACEField",
    device=DEVICE,
    default_dtype="float64",
    electric_field=[0.0, 0.0, 0.0],
    compute_polarization=False,
    compute_becs=False,
    compute_polarizability=False,
)
atoms.calc = relax_calc
print("Starting fixed-cell BFGS relaxation; optimizer step, energy, and maximum force will print below.", flush=True)
BFGS(atoms, logfile="-").run(fmax=0.05, steps=60)
print("atoms:", len(atoms), "| volume (Å³):", atoms.get_volume())
print("maximum force after fixed-cell relaxation (eV/Å):",
      float(np.linalg.norm(atoms.get_forces(), axis=1).max()))


## Evaluate electronic susceptibility and BECs

The calculator returns polarization in e/Å², BECs in e, and the volume-normalized electronic susceptibility tensor. Add the identity tensor to susceptibility to obtain $\epsilon_\infty$.


In [ ]:
response_calc = MACECalculator(
    model_paths=str(MODEL),
    model_type="MACEField",
    device=DEVICE,
    default_dtype="float64",
    electric_field=[0.0, 0.0, 0.0],
    compute_polarization=True,
    compute_becs=True,
    compute_polarizability=True,
)
atoms.calc = response_calc
atoms.get_potential_energy()
polarization = np.asarray(response_calc.results["polarization"], dtype=float)
becs = np.asarray(response_calc.results["becs"], dtype=float).reshape(len(atoms), 3, 3)
chi_electronic = np.asarray(
    response_calc.results["polarizability"], dtype=float
).reshape(3, 3)
epsilon_infinity = np.eye(3) + chi_electronic

print("polarization (e/Å²):", polarization)
print("BEC tensor shape:", becs.shape)
print("mean BEC over atoms (e):")
print(becs.mean(axis=0))
print("electronic susceptibility:")
print(chi_electronic)
print("epsilon_infinity:")
print(epsilon_infinity)


## Differentiate the field-conditioned energy twice with respect to positions

The current ASE calculator exposes `get_hessian(atoms)` for MACEField and returns an array with shape `(3N, N, 3)`. This zero-field example reshapes it to the dense Cartesian `(3N, 3N)` matrix in eV/Å² and symmetrizes numerical noise. The current helper does not forward a nonzero `calculator.electric_field` into this calculation, so the explicit zero-field setting here is important; do not use this call for a field-dependent Hessian.

In [ ]:
print("Calculating the dense 216 x 216 Cartesian Hessian with MACECalculator.get_hessian...", flush=True)
hessian = np.asarray(relax_calc.get_hessian(atoms)).reshape(3 * len(atoms), -1)
hessian = 0.5 * (hessian + hessian.T)
print("Hessian calculation complete.", flush=True)
print("Hessian shape:", hessian.shape, "| units: eV/Å²")
print("maximum asymmetry after symmetrization:",
      float(np.max(np.abs(hessian - hessian.T))))


## Combine BECs and Hessian for the harmonic ionic response

At fixed cell, the static harmonic displacement response is $\chi_\mathrm{ion}=Z H^+ Z^T/(\Omega\epsilon_0)$, where $H^+$ is the pseudoinverse after removing translational zero modes. Here $\epsilon_0$ is expressed in the matching e/(V Å) unit convention. The calculation excludes negative and near-zero Hessian eigenvalues; any additional negative modes indicate an unstable structure and make the harmonic dielectric estimate unreliable.


In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(hessian)
cutoff = max(float(np.max(np.abs(eigenvalues))) * 1.0e-8, 1.0e-8)
stable_modes = eigenvalues > cutoff
negative_modes = int(np.count_nonzero(eigenvalues < -cutoff))
hessian_pinv = (
    eigenvectors[:, stable_modes] / eigenvalues[stable_modes]
) @ eigenvectors[:, stable_modes].T

# Z[polarization direction, Cartesian displacement coordinate]
z_matrix = np.transpose(becs, (1, 0, 2)).reshape(3, -1)
eps0_e_per_v_angstrom = 8.8541878128e-12 / 1.602176634e-19 / 1.0e10
chi_ionic = (
    z_matrix @ hessian_pinv @ z_matrix.T
    / (atoms.get_volume() * eps0_e_per_v_angstrom)
)
epsilon_static_harmonic = epsilon_infinity + chi_ionic

print("Hessian eigenvalues below the acoustic cutoff:", negative_modes)
print("retained positive modes:", int(np.count_nonzero(stable_modes)))
print("ionic susceptibility:")
print(chi_ionic)
print("harmonic static dielectric tensor:")
print(epsilon_static_harmonic)
if negative_modes:
    print("Warning: the relaxed model structure has unstable modes; do not interpret this harmonic ionic estimate as a stable dielectric constant.")


## 3. Zero-field MD and response spectra at 300 K

Every MD step prints time, target and instantaneous temperature, energy, polarization, and mean electronic susceptibility. It writes an ASE trajectory and displays per-step time-series plots and an animation. The following trajectory uses a Langevin thermostat at 300 K and zero applied electric field. Polarization and electronic susceptibility are sampled along the trajectory to form autocorrelation functions and exploratory dielectric, IR, and Raman spectra.


## Run zero-field molecular dynamics

The short default is `MD_STEPS = 1,000` (0.5 ps at 0.5 fs), which gives a coarse raw frequency spacing near 67 cm⁻¹. Increase `MD_STEPS` in the MD cell, or set `MACEFIELD_SIO2_MD_STEPS`, for longer trajectories; for example, 20,000 steps is 10 ps and takes roughly 20 times longer than the default. Runtime depends on the device and selected model, and the cell prints a rough estimate after the first 20 steps. Longer trajectories improve spectral resolution but still need block and initial-condition checks.

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from ase import units
from ase.io import read, Trajectory
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.optimize import BFGS
from mace.calculators import MACECalculator
import torch

MD_MODEL_SOURCE = "provided"  # Choose: "provided", "trained", or "mh1"
MD_MODEL_PATH, MD_MODEL_HEAD = select_md_model(
    MD_MODEL_SOURCE, DATA_DIR / "sio2/MACE-field-SiO2.model"
)
print(
    "MD model:", MD_MODEL_PATH,
    "| source:", MD_MODEL_SOURCE,
    "| head:", MD_MODEL_HEAD or "model default",
    flush=True,
)
SIO2_DATA = DATA_DIR / "sio2/SiO2-preprocessed.extxyz"
atoms = read(SIO2_DATA, index=0)
atoms.info.clear()
for key in list(atoms.arrays):
    if key not in {"numbers", "positions"}:
        atoms.arrays.pop(key)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
md_calculator_options = {
    "model_paths": str(MD_MODEL_PATH),
    "model_type": "MACEField",
    "device": DEVICE,
    "default_dtype": "float32",
    "electric_field": [0.0, 0.0, 0.0],
}
if MD_MODEL_HEAD is not None:
    md_calculator_options["head"] = MD_MODEL_HEAD
relax_calc = MACECalculator(
    **md_calculator_options,
    compute_polarization=False,
    compute_becs=False,
    compute_polarizability=False,
)
atoms.calc = relax_calc
print("Starting fixed-cell BFGS relaxation before MD; optimizer step, energy, and maximum force will print below.", flush=True)
BFGS(atoms, logfile="-").run(fmax=0.05, steps=40)

temperature_K = 300.0
dt_fs = 0.5
MD_STEPS = int(os.environ.get("MACEFIELD_SIO2_MD_STEPS", "1000"))
if MD_STEPS < 1:
    raise ValueError("MD_STEPS must be at least 1")
n_steps = MD_STEPS
rng = np.random.default_rng(31)
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K, rng=rng)
Stationary(atoms)

calc = MACECalculator(
    **md_calculator_options,
    compute_polarization=True,
    compute_becs=False,
    compute_polarizability=True,
)
atoms.calc = calc
dyn = Langevin(
    atoms,
    timestep=dt_fs * units.fs,
    temperature_K=temperature_K,
    friction=0.02 / units.fs,
    rng=rng,
)

MD_DIR = TUTORIAL_DIR / "runs" / f"sio2_md_300K_{time.strftime('%Y%m%d_%H%M%S')}"
MD_DIR.mkdir(parents=True, exist_ok=False)
trajectory_path = MD_DIR / "sio2_300K_zero_field.traj"
trajectory = Trajectory(str(trajectory_path), "w", atoms)
polarization_series = []
susceptibility_series = []
energy_series = []
temperature_series = []
time_series = []
position_series = []

def sample_response(step):
    energy = float(atoms.get_potential_energy())
    polarization = np.asarray(calc.results["polarization"], dtype=float).copy()
    susceptibility = np.asarray(calc.results["polarizability"], dtype=float).reshape(3, 3).copy()
    temperature_now = float(atoms.get_temperature())
    polarization_series.append(polarization)
    susceptibility_series.append(susceptibility)
    energy_series.append(energy)
    temperature_series.append(temperature_now)
    time_series.append(step * dt_fs)
    position_series.append(atoms.positions.copy())
    trajectory.write(atoms)
    print(
        f"MD step {step:04d}/{n_steps} | t={step * dt_fs:7.1f} fs "
        f"| Ttarget={temperature_K:.1f} K | Tinst={temperature_now:7.2f} K "
        f"| E={energy:.6f} eV | P=({polarization[0]:+.4e}, {polarization[1]:+.4e}, {polarization[2]:+.4e}) e/Å² "
        f"| mean chi={np.trace(susceptibility) / 3:+.5f}",
        flush=True,
    )
    if step == min(20, n_steps):
        estimated_seconds = (time.perf_counter() - md_started) * n_steps / step
        print(
            f"Rough MD runtime estimate from the first {step} steps: "
            f"{estimated_seconds / 60:.1f} min; actual time may vary with startup and hardware.",
            flush=True,
        )

print(
    f"Starting zero-field SiO2 MD at {temperature_K:.1f} K: {n_steps} steps "
    f"({n_steps * dt_fs / 1000:.2f} ps), dt={dt_fs:g} fs, device={DEVICE}.", flush=True
)
print(
    "Runtime warning: the fixed-cell relaxation runs first; MD time depends on the "
    "selected model and device. A rough MD estimate prints after the first 20 steps.",
    flush=True,
)
md_started = time.perf_counter()
dyn.attach(lambda: sample_response(dyn.get_number_of_steps()), interval=1)
dyn.run(n_steps)
trajectory.close()

polarization_series = np.asarray(polarization_series)
susceptibility_series = np.asarray(susceptibility_series)
energy_series = np.asarray(energy_series)
temperature_series = np.asarray(temperature_series)
position_series = np.asarray(position_series)
time_fs = np.asarray(time_series, dtype=float)
volume_A3 = atoms.get_volume()
print("trajectory saved to:", trajectory_path)
print("device:", DEVICE)
print("temperature (K):", temperature_K)
print("electric field (V/Å):", [0.0, 0.0, 0.0])
print("frames:", len(time_fs), "| duration (ps):", time_fs[-1] / 1000)
print("frequency spacing (cm⁻¹):",
      1.0 / (len(time_fs) * dt_fs * 1.0e-15) / (299792458.0 * 100.0))


## Visualize the 300 K SiO₂ trajectory

These plots include every saved MD frame. The animation shows the periodic cell and the Si/O atom positions; the corresponding `.traj` file is saved under `tutorials/macefield/runs/`.


In [ ]:
from itertools import product
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for component, label in enumerate(("Px", "Py", "Pz")):
    axes[0, 0].plot(time_fs, polarization_series[:, component], label=label)
axes[0, 0].set(xlabel="time (fs)", ylabel="polarization (e/Å²)", title="Polarization trajectory")
axes[0, 0].legend()
axes[0, 1].plot(time_fs, temperature_series, color="#e45756")
axes[0, 1].axhline(temperature_K, color="black", linestyle="--", linewidth=1)
axes[0, 1].set(xlabel="time (fs)", ylabel="instantaneous T (K)", title="300 K thermostat trajectory")
axes[1, 0].plot(time_fs, energy_series, color="#4c78a8")
axes[1, 0].set(xlabel="time (fs)", ylabel="potential energy (eV)", title="Energy trajectory")
chi_iso = np.trace(susceptibility_series, axis1=1, axis2=2) / 3.0
axes[1, 1].plot(time_fs, chi_iso, color="#8c62aa")
axes[1, 1].set(xlabel="time (fs)", ylabel="mean electronic susceptibility", title="Electronic response trajectory")
for ax in axes.flat:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

cell = np.asarray(atoms.cell.array)
symbols = np.asarray(atoms.get_chemical_symbols())
corners = np.asarray([np.asarray(bits, dtype=float) @ cell for bits in product((0, 1), repeat=3)])
frame_stride = max(1, int(np.ceil(len(position_series) / 60)))
frame_ids = list(range(0, len(position_series), frame_stride))
if frame_ids[-1] != len(position_series) - 1:
    frame_ids.append(len(position_series) - 1)
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
colors = {"Si": "#f2b134", "O": "#d84b4b"}

def wrapped_sio2_positions(frame):
    frac = position_series[frame] @ np.linalg.inv(cell)
    frac -= np.floor(frac)
    return frac @ cell

def draw_sio2_cell():
    for bits in product((0, 1), repeat=3):
        start = np.asarray(bits, dtype=float) @ cell
        for axis in range(3):
            if not bits[axis]:
                end = start + cell[axis]
                ax.plot(*np.vstack([start, end]).T, color="0.55", linewidth=0.8)

def update_sio2(frame):
    ax.clear()
    positions = wrapped_sio2_positions(frame)
    for element in np.unique(symbols):
        mask = symbols == element
        ax.scatter(*positions[mask].T, s=36 if element == "Si" else 16,
                   color=colors.get(str(element), "gray"), label=str(element), depthshade=True)
    draw_sio2_cell()
    limits = np.column_stack([corners.min(axis=0), corners.max(axis=0)])
    ax.set_xlim(*limits[0]); ax.set_ylim(*limits[1]); ax.set_zlim(*limits[2])
    ax.set_box_aspect(np.maximum(np.ptp(corners, axis=0), 1.0e-6))
    ax.set_xlabel("x (Å)"); ax.set_ylabel("y (Å)"); ax.set_zlabel("z (Å)")
    ax.set_title(f"300 K SiO₂ | MD step {frame} | t={time_fs[frame]:.1f} fs")
    ax.legend(loc="upper right", fontsize=8)
    return []

animation = FuncAnimation(fig, update_sio2, frames=frame_ids, interval=180, blit=False)
plt.close(fig)
display(HTML(animation.to_jshtml(fps=5)))


## Build polarization and polarizability autocorrelation functions

The plots above show the trajectory before this cell computes its correlations. Polarization is recorded in e/Å². The calculator's volume-normalized susceptibility is dimensionless and proportional to the extensive cell polarizability at this fixed volume. The spectra below are normalized for shape, not absolute scattering cross sections.


In [ ]:
def autocorrelation_matrix(series):
    values = np.asarray(series, dtype=float)
    values = values - values.mean(axis=0, keepdims=True)
    n = len(values)
    n_fft = 1 << (2 * n - 1).bit_length()
    spectrum = np.fft.rfft(values, n=n_fft, axis=0)
    denominator = np.arange(n, 0, -1, dtype=float)
    correlation = np.empty((n, values.shape[1], values.shape[1]))
    for i in range(values.shape[1]):
        for j in range(values.shape[1]):
            raw = np.fft.irfft(
                spectrum[:, i] * spectrum[:, j].conj(), n=n_fft
            )[:n]
            correlation[:, i, j] = raw / denominator
    return correlation

def autocorrelation_scalar(series):
    values = np.asarray(series, dtype=float)
    values = values - values.mean()
    n = len(values)
    n_fft = 1 << (2 * n - 1).bit_length()
    spectrum = np.fft.rfft(values, n=n_fft)
    raw = np.fft.irfft(spectrum * spectrum.conj(), n=n_fft)[:n]
    return raw / np.arange(n, 0, -1, dtype=float)

p_corr = autocorrelation_matrix(polarization_series)
lag_fs = np.arange(len(p_corr)) * dt_fs
alpha = susceptibility_series
alpha_iso = np.trace(alpha, axis1=1, axis2=2) / 3.0
alpha_aniso_components = np.column_stack(
    [
        (alpha[:, 0, 0] - alpha[:, 1, 1]) / np.sqrt(2.0),
        (alpha[:, 1, 1] - alpha[:, 2, 2]) / np.sqrt(2.0),
        (alpha[:, 2, 2] - alpha[:, 0, 0]) / np.sqrt(2.0),
        np.sqrt(3.0) * alpha[:, 0, 1],
        np.sqrt(3.0) * alpha[:, 1, 2],
        np.sqrt(3.0) * alpha[:, 2, 0],
    ]
)
alpha_iso_corr = autocorrelation_scalar(alpha_iso)
alpha_aniso_corr = sum(
    autocorrelation_scalar(alpha_aniso_components[:, i])
    for i in range(alpha_aniso_components.shape[1])
)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for axis in range(3):
    denom = max(float(p_corr[0, axis, axis]), 1.0e-30)
    axes[0].plot(lag_fs / 1000, p_corr[:, axis, axis] / denom,
                 label=f"component {axis}")
axes[0].set(xlabel="lag (ps)", ylabel="normalized ACF",
            title="Polarization autocorrelation")
axes[0].legend()
axes[1].plot(lag_fs / 1000,
             alpha_iso_corr / max(float(alpha_iso_corr[0]), 1.0e-30),
             label="isotropic susceptibility")
axes[1].plot(lag_fs / 1000,
             alpha_aniso_corr / max(float(alpha_aniso_corr[0]), 1.0e-30),
             label="anisotropic tensor")
axes[1].set(xlabel="lag (ps)", ylabel="normalized ACF",
            title="Polarizability autocorrelation")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## Estimate the frequency-dependent dielectric tensor and spectra

The one-sided fluctuation-dissipation transform is $\epsilon(\omega)=\epsilon_\infty+V/(\epsilon_0 k_B T)[C(0)-i\omega\tilde C(\omega)]$, using the Fourier convention $e^{-i\omega t}$. The lower-right panel marks the trajectory static limit $\epsilon(0)$ and electronic limit $\epsilon_\infty$ with horizontal dashed lines. The static line uses the zero-frequency value of this same finite-temperature MD estimate. For readability, plotted spectra use 12 cm$^{-1}$ Gaussian broadening; the saved CSV and NPZ retain unbroadened values. The IR and Raman curves are relative classical spectra; the Raman proxy omits incident-laser, Bose-population, and absolute-intensity factors.

In [ ]:
n = len(time_fs)
n_fft = 1 << (2 * n - 1).bit_length()
window = 0.5 * (
    1.0 + np.cos(np.pi * lag_fs / (lag_fs[-1] + dt_fs))
)
p_corr_windowed = p_corr * window[:, None, None]
p_transform = np.fft.rfft(p_corr_windowed, n=n_fft, axis=0) * dt_fs

frequency_hz = np.fft.rfftfreq(n_fft, d=dt_fs * 1.0e-15)
frequency_cm = frequency_hz / (299792458.0 * 100.0)
omega_rad_fs = 2.0 * np.pi * np.fft.rfftfreq(n_fft, d=dt_fs)

elementary_charge = 1.602176634e-19
epsilon0_SI = 8.8541878128e-12
boltzmann = 1.380649e-23
polarization_unit_SI = elementary_charge / 1.0e-20
volume_m3 = volume_A3 * 1.0e-30
fluctuation_prefactor = (
    volume_m3 * polarization_unit_SI**2
    / (epsilon0_SI * boltzmann * temperature_K)
)
epsilon_infinity = np.eye(3) + np.mean(susceptibility_series, axis=0)
epsilon_omega = (
    epsilon_infinity[None, :, :]
    + fluctuation_prefactor
    * (
        p_corr[0][None, :, :]
        - 1j * omega_rad_fs[:, None, None] * p_transform
    )
)

alpha_iso_transform = np.fft.rfft(
    alpha_iso_corr * window, n=n_fft
) * dt_fs
alpha_aniso_transform = np.fft.rfft(
    alpha_aniso_corr * window, n=n_fft
) * dt_fs
ir_proxy = frequency_cm**2 * np.maximum(
    np.trace(p_transform.real, axis1=1, axis2=2), 0.0
)
raman_proxy = frequency_cm**2 * (
    45.0 * np.maximum(alpha_iso_transform.real, 0.0)
    + 7.0 * np.maximum(alpha_aniso_transform.real, 0.0)
)

from scipy.ndimage import gaussian_filter1d

# This display-only broadening smooths the zero-padded grid without changing saved raw spectra.
spectrum_broadening_fwhm_cm = 12.0
frequency_step_cm = float(frequency_cm[1] - frequency_cm[0])
spectrum_sigma_bins = (
    spectrum_broadening_fwhm_cm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    / frequency_step_cm
)
epsilon_plot = (
    gaussian_filter1d(epsilon_omega.real, spectrum_sigma_bins, axis=0, mode="nearest")
    + 1j * gaussian_filter1d(epsilon_omega.imag, spectrum_sigma_bins, axis=0, mode="nearest")
)
epsilon_plot[0] = epsilon_omega[0]  # keep the plotted static limit exact
ir_plot = gaussian_filter1d(ir_proxy, spectrum_sigma_bins, mode="nearest")
raman_plot = gaussian_filter1d(raman_proxy, spectrum_sigma_bins, mode="nearest")

show = frequency_cm <= 1600.0
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for i in range(3):
    axes[0, 0].plot(frequency_cm[show], epsilon_plot.real[show, i, i],
                    label=f"epsilon {i}{i}")
    axes[0, 0].plot(frequency_cm[show], -epsilon_plot.imag[show, i, i],
                    linestyle="--", label=f"loss {i}{i}")
axes[0, 0].set(xlabel="wavenumber (cm⁻¹)", ylabel="relative permittivity",
              title=f"Frequency-dependent dielectric tensor ({spectrum_broadening_fwhm_cm:g} cm⁻¹ broadening)")
axes[0, 0].legend(fontsize=7)
axes[0, 1].plot(frequency_cm[show], ir_plot[show], color="#2f6b8a")
axes[0, 1].set(xlabel="wavenumber (cm⁻¹)", ylabel="relative intensity",
               title="IR proxy from polarization fluctuations")
axes[1, 0].plot(frequency_cm[show], raman_plot[show], color="#9b4f96")
axes[1, 0].set(xlabel="Raman shift (cm⁻¹)", ylabel="relative intensity",
               title="Raman proxy from susceptibility fluctuations")
axes[1, 1].plot(frequency_cm[show], np.trace(epsilon_plot.real[show], axis1=1, axis2=2) / 3,
                label="mean Re epsilon")
axes[1, 1].plot(frequency_cm[show], -np.trace(epsilon_plot.imag[show], axis1=1, axis2=2) / 3,
                label="mean dielectric loss")
static_limit_mean = float(np.trace(epsilon_omega[0].real) / 3.0)
electronic_limit_mean = float(np.trace(epsilon_infinity) / 3.0)
axes[1, 1].axhline(
    static_limit_mean, color="black", linestyle="--", linewidth=1.2,
    label=f"static limit ε(0) = {static_limit_mean:.2f}",
)
axes[1, 1].axhline(
    electronic_limit_mean, color="0.45", linestyle="--", linewidth=1.4,
    label=f"electronic limit ε∞ = {electronic_limit_mean:.2f}",
)
axes[1, 1].set(xlabel="wavenumber (cm⁻¹)", ylabel="relative permittivity",
               title=f"Mean dielectric response ({spectrum_broadening_fwhm_cm:g} cm⁻¹ broadening)")
axes[1, 1].legend()
for ax in axes.flat:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

print("static fluctuation limit, diagonal:", np.diag(epsilon_omega[0].real))
print("electronic limit epsilon infinity, diagonal:", np.diag(epsilon_infinity))
print(f"mean static limit: {static_limit_mean:.6g}")
print(f"mean electronic limit: {electronic_limit_mean:.6g}")
print(f"Zero-padding interpolates the grid; the {time_fs[-1] / 1000:.3f} ps trajectory sets the physical resolution.")


## Convergence notes

The MD is fixed-cell, classical, and at zero field. The 10 ps GPU trajectory improves frequency resolution, and plots apply a 12 cm⁻¹ Gaussian display broadening; raw spectra remain saved separately. Reliable spectra still need checks over independent initial velocities and time blocks, timestep and thermostat sensitivity, and model validation over the sampled configurations. The dielectric estimate also depends on the periodic boundary convention and finite-size polarization fluctuations.

For methods context, see the [MACEField paper](https://doi.org/10.1038/S41467-025-59304-1) and the [frequency-dependent dielectric function tutorial in Calorine](https://calorine.materialsmodeling.org/get_started/dielectric_functions.html).
